# Mini-TP 4 — Puntúa tu modelo sobre un flujo (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Aplica **tu modelo** (el de la Sesión 1) sobre un **flujo de eventos** en lugar de pedidos sueltos. Completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un **flujo** de eventos con las *features* de tu modelo (simulado, o consumido de Kafka).
2. Un **consumidor** que puntúe tu modelo sobre **cada** evento (inferencia online).
3. **Métricas por ventana:** throughput (ev/s), latencia **p95** y un indicador simple de **drift** de entradas.
4. Una **alerta** cuando una métrica cruce un umbral.
5. Una **comparación** contra puntuar el mismo lote en **batch**, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; inferencia online sobre el flujo; métricas por ventana correctas; y la reflexión streaming vs batch.

> Apóyate en `streaming_tutorial.ipynb`. Entorno uv: `uv add scikit-learn joblib numpy kafka-python`.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "numpy"])
print("Listo.")

Listo.


## 1. Tu modelo

Carga **tu** modelo. Si aún no lo tienes a mano, el placeholder deja probar la mecánica.

In [2]:
import joblib, numpy as np
import sys; sys.path.append("../tp1")   # el pipeline usa preprocesamiento.py del TP1

modelo = joblib.load("../tp1/modelo/modelo.joblib")["pipeline"]
N_FEATURES = 10
print("Modelo cargado. n_features =", N_FEATURES)

Modelo cargado. n_features = 10


## 2. El flujo

Simula un flujo con las features de tu modelo. Introduce un cambio de distribución a mitad de camino para poder detectar drift. (Alternativa: consumir de Kafka como en el tutorial.)

In [3]:
import queue, threading, time, random
import pandas as pd
stream = queue.Queue(); STOP = object()
N = 800

# pacientes reales del dataset
pacientes = pd.read_csv("../tp2/dataset/healthcare-dataset-stroke-data.csv").drop(columns=["id", "stroke"])

def productor():
    for i in range(N):
        paciente = pacientes.iloc[random.randrange(len(pacientes))].to_dict()
        if i >= N//2:
            paciente["avg_glucose_level"] += 50      # drift a mitad de camino
        stream.put({"values": paciente, "t": time.time()})
        time.sleep(0.02)
    stream.put(STOP)
print("Productor listo.")

Productor listo.


## 3. Consumidor con inferencia online + métricas por ventana

In [4]:
from collections import deque
import numpy as np

latencias = []; ventana = deque(); alertas = []
eventos = []   # guardo cada paciente para puntuarlo en batch en la celda 4

def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP: break
        eventos.append(ev["values"])
        t0 = time.perf_counter()
        pred = modelo.predict(pd.DataFrame([ev["values"]]))[0]
        latencias.append((time.perf_counter() - t0) * 1000)

        ventana.append((ev["t"], ev["values"]["avg_glucose_level"], pred))
        ahora = ev["t"]
        while ventana and ahora - ventana[0][0] > 1.0:
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:
            media_glucosa = float(np.mean([w[1] for w in ventana]))
            # drift: media de glucosa en la ventana (en el dataset es aprox 106)
            if media_glucosa > 130:
                alertas.append((procesados, media_glucosa))

t0 = time.time()
pt = threading.Thread(target=productor); ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"throughput : {len(latencias)/dur:.0f} ev/s")
print(f"p95        : {np.percentile(latencias, 95):.3f} ms")
print(f"alertas de drift: {alertas}")

throughput : 48 ev/s
p95        : 7.942 ms
alertas de drift: [(600, 169.37020408163266), (800, 149.5273469387755)]


## 4. Comparación contra batch

Puntúa el mismo conjunto de eventos de una sola vez (batch) y compara tiempo/uso frente al streaming.

In [5]:
LOTE = pd.DataFrame(eventos)  

t0 = time.perf_counter()
batch_pred = modelo.predict(LOTE)
dur_batch = (time.perf_counter() - t0) * 1000

print(f"batch     : {len(LOTE)} eventos en {dur_batch:.1f} ms ({dur_batch/len(LOTE):.3f} ms por evento)")
print(f"streaming : {len(latencias)} eventos en {sum(latencias):.1f} ms de scoring ({np.mean(latencias):.3f} ms por evento)")

batch     : 800 eventos en 7.6 ms (0.009 ms por evento)
streaming : 800 eventos en 5473.6 ms de scoring (6.842 ms por evento)


### Reflexión

Con el streaming se logra dar una predicción de cada paciente apenas llega, cosa que con batch no es posible ya que se obtiene una vez que se juntó todo el lote. También se pudo detectar el drift apenas ocurrió. Sin embargo, con batch fue más rápido predecir (800 pacientes en 7.6 ms contra los 5473.6 ms de streming).

En mi plataforma podría usar streaming para vigilar el drift sobre los datos que se van ingresando. En cuanto a batch, se podría usar para reentrenar. Cuando se logre tener una determinada cantidad de pacientes (un lote), se puede reentrenar el modelo y compararlo con el que está. Si es mejor, ese pasa a ser el nuevo modelo de producción.

El drift lo detecté con la media de la glucosa (variable elegida como ejemplo) en una ventana deslizante de 1s, comparada contra la media del dataset (106), con un umbral de 130. Al dispararse la alerta se podrían revisar los datos que están llegando y si ese cambio se sostiene en el tiempo, podría ser un indicador de que es necesario reentrenar.

## 5. (Opcional) Con Kafka en Docker

Levanta un broker y reemplaza la cola por un topic real:
```bash
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false
uv add kafka-python
```
Produce a un topic `eventos` y consume puntuando tu modelo (ver la parte 6 del tutorial).